In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

In [2]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
llm=ChatOpenAI()

In [ ]:
def chat_node(state: ChatState):
    messages= state["messages"]

    response=llm.invoke(messages)

    return {"messages": [response]}

In [ ]:
checkpointer=MemorySaver()

mygraph=StateGraph(ChatState)

mygraph.add_node("chat_node", chat_node)

mygraph.add_edge(START, "chat_node")
mygraph.add_edge("chat_node", END)

chatbot=mygraph.compile(checkpointer=checkpointer)

In [ ]:
initial_state={
    "message": [HumanMessage(content="What is the capital of india")]
}

chatbot.invoke(initial_state)

In [ ]:

thread_id="1"

while True:
    user_message=input("Type here:")

    if user_message.strip().lower() in ["exit", "quit", "bye"]:
        break
    config = {"configurable": {"thread_id": thread_id}}
    chatbot.invoke({"message": [HumanMessage(content=user_message)]}, config=config)
        

